# 05 - VEP consequence annotation local

Notebook nay tao annotation `consequence` bang Ensembl VEP local qua Docker.

Input:

- `processed/clinvar_training_metadata.parquet`
- `processed/clinvar_for_vep.vcf`

Output:

- `processed/clinvar_vep_consequence.tsv`
- `Data/annotations/variant_annotations.csv`

Ghi chu: buoc nay chi tao `consequence`. Cac cot `gnomAD_AF`, `CADD_PHRED`, `SpliceAI_max` de trong de join bang database rieng sau.

In [3]:
from pathlib import Path
import os
import subprocess

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
PROCESSED_DIR = PROJECT_DIR / "processed"
ANNOTATION_DIR = PROJECT_DIR / "Data" / "annotations"
VEP_CACHE_DIR = PROJECT_DIR / "Data" / "vep_cache"

METADATA_PATH = PROCESSED_DIR / "clinvar_training_metadata.parquet"
VCF_PATH = PROCESSED_DIR / "clinvar_for_vep.vcf"
VEP_TSV_PATH = PROCESSED_DIR / "clinvar_vep_consequence.tsv"
ANNOTATION_CSV_PATH = ANNOTATION_DIR / "variant_annotations.csv"

ANNOTATION_DIR.mkdir(exist_ok=True)
VEP_CACHE_DIR.mkdir(exist_ok=True)

for required_path in [METADATA_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

PROJECT_DIR, METADATA_PATH, VCF_PATH, VEP_TSV_PATH, ANNOTATION_CSV_PATH

(PosixPath('/mnt/MyData/Bioinformatics/Project'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/clinvar_training_metadata.parquet'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/clinvar_for_vep.vcf'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/processed/clinvar_vep_consequence.tsv'),
 PosixPath('/mnt/MyData/Bioinformatics/Project/Data/annotations/variant_annotations.csv'))

## 1. Tao VCF dau vao cho VEP

VCF dung ID on dinh theo key:

`Chromosome:PositionVCF:ReferenceAlleleVCF:AlternateAlleleVCF`

Neu file `processed/clinvar_for_vep.vcf` da ton tai, cell nay se ghi de de dam bao dong bo voi metadata hien tai.

In [4]:
key_cols = [
    "Chromosome",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
]

variant_df = pd.read_parquet(METADATA_PATH, columns=key_cols).copy()
for col in key_cols:
    variant_df[col] = variant_df[col].astype(str)

valid_snv_mask = (
    variant_df["ReferenceAlleleVCF"].str.fullmatch("[ACGT]")
    & variant_df["AlternateAlleleVCF"].str.fullmatch("[ACGT]")
)
if not valid_snv_mask.all():
    raise ValueError(f"Co {(~valid_snv_mask).sum():,} variants khong phai SNV A/C/G/T")

variant_df = variant_df.drop_duplicates(subset=key_cols).copy()
variant_df["PositionVCF_int"] = variant_df["PositionVCF"].astype(int)
variant_df = variant_df.sort_values([
    "Chromosome", "PositionVCF_int", "ReferenceAlleleVCF", "AlternateAlleleVCF"
])

with VCF_PATH.open("w") as handle:
    handle.write("##fileformat=VCFv4.2\n")
    handle.write("##source=clinvar_training_metadata_for_vep\n")
    handle.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
    for row in variant_df.itertuples(index=False):
        variant_id = f"{row.Chromosome}:{row.PositionVCF}:{row.ReferenceAlleleVCF}:{row.AlternateAlleleVCF}"
        handle.write(
            f"{row.Chromosome}\t{row.PositionVCF}\t{variant_id}\t"
            f"{row.ReferenceAlleleVCF}\t{row.AlternateAlleleVCF}\t.\t.\t.\n"
        )

print(f"Saved VCF: {VCF_PATH}")
print(f"Variants: {len(variant_df):,}")
print(f"VCF size MB: {VCF_PATH.stat().st_size / 1024 / 1024:.2f}")

Saved VCF: /mnt/MyData/Bioinformatics/Project/processed/clinvar_for_vep.vcf
Variants: 460,488
VCF size MB: 16.40


In [5]:
# Kiem tra nhanh VCF header va so dong variant.
with VCF_PATH.open("r") as handle:
    preview = [next(handle).rstrip("\n") for _ in range(8)]

variant_line_count = sum(1 for line in VCF_PATH.open("r") if not line.startswith("#"))

print("\n".join(preview))
print(f"variant_line_count={variant_line_count:,}")
assert variant_line_count == len(variant_df)

##fileformat=VCFv4.2
##source=clinvar_training_metadata_for_vep
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
1	69134	1:69134:A:G	A	G	.	.	.
1	924518	1:924518:G:C	G	C	.	.	.
1	925956	1:925956:C:T	C	T	.	.	.
1	925969	1:925969:C:T	C	T	.	.	.
1	925980	1:925980:C:T	C	T	.	.	.
variant_line_count=460,488


## 2. Kiem tra Docker va VEP cache

VEP cache GRCh38 co the rat lon va mat thoi gian tai. Cell nay chi kiem tra trang thai, chua tai cache.

In [6]:
def run_command(command, check=True):
    print("$", " ".join(str(part) for part in command), flush=True)
    completed = subprocess.run(command, check=False, text=True)
    if check and completed.returncode != 0:
        raise subprocess.CalledProcessError(completed.returncode, command)
    return completed


docker_version = run_command(["docker", "--version"], check=True)
print(f"VEP cache dir: {VEP_CACHE_DIR}")
print("Cache dir contents:")
for path in sorted(VEP_CACHE_DIR.glob("*"))[:20]:
    print("-", path)

$ docker --version
Docker version 29.4.2, build 055a478VEP cache dir: /mnt/MyData/Bioinformatics/Project/Data/vep_cache
Cache dir contents:



## 3. Cai/download VEP GRCh38 cache

Cell nay mac dinh **khong tu chay** de tranh tai cache lon ngoai y muon.

Dat `RUN_CACHE_INSTALL = True` roi chay cell de tai cache.

In [7]:
RUN_CACHE_INSTALL = True

# Jupyter khong co TTY that, nen khong dung `-t -i` trong docker run.
# Quan trong: phai co --CACHEDIR /data de cache duoc ghi vao thu muc mounted Data/vep_cache.
cache_install_command = [
    "docker", "run", "--rm",
    "-v", f"{VEP_CACHE_DIR}:/data",
    "ensemblorg/ensembl-vep",
    "INSTALL.pl",
    "--AUTO", "cf",
    "--SPECIES", "homo_sapiens",
    "--ASSEMBLY", "GRCh38",
    "--CACHEDIR", "/data",
]

print("Cache install command:")
print(" ".join(cache_install_command))

if RUN_CACHE_INSTALL:
    run_command(cache_install_command, check=True)
else:
    print("RUN_CACHE_INSTALL=False; chua tai cache. Doi thanh True neu can cai VEP cache.")

Cache install command:
docker run --rm -v /mnt/MyData/Bioinformatics/Project/Data/vep_cache:/data ensemblorg/ensembl-vep INSTALL.pl --AUTO cf --SPECIES homo_sapiens --ASSEMBLY GRCh38 --CACHEDIR /data
$ docker run --rm -v /mnt/MyData/Bioinformatics/Project/Data/vep_cache:/data ensemblorg/ensembl-vep INSTALL.pl --AUTO cf --SPECIES homo_sapiens --ASSEMBLY GRCh38 --CACHEDIR /data
 - getting list of available cache files
 - downloading https://ftp.ensembl.org/pub/release-115/variation/indexed_vep_cache/homo_sapiens_vep_115_GRCh38.tar.gz


CalledProcessError: Command '['docker', 'run', '--rm', '-v', '/mnt/MyData/Bioinformatics/Project/Data/vep_cache:/data', 'ensemblorg/ensembl-vep', 'INSTALL.pl', '--AUTO', 'cf', '--SPECIES', 'homo_sapiens', '--ASSEMBLY', 'GRCh38', '--CACHEDIR', '/data']' returned non-zero exit status 137.

## 4. Chay VEP consequence annotation

Sau khi cache san sang, dat `RUN_VEP = True` de chay VEP tren `processed/clinvar_for_vep.vcf`.

Output TSV se la `processed/clinvar_vep_consequence.tsv`.

In [ ]:
RUN_VEP = True

expected_cache_dirs = list(VEP_CACHE_DIR.glob("homo_sapiens*GRCh38*")) + list(VEP_CACHE_DIR.glob("homo_sapiens/*GRCh38*"))
cache_ready = any(path.is_dir() for path in expected_cache_dirs)

print(f"Cache ready: {cache_ready}")
if not cache_ready:
    print("Khong thay VEP GRCh38 cache trong Data/vep_cache.")
    print("Hay chay cell RUN_CACHE_INSTALL=True truoc. Cache dir hien tai:")
    for path in sorted(VEP_CACHE_DIR.glob("*"))[:30]:
        print("-", path)

vep_command = [
    "docker", "run", "--rm",
    "-v", f"{VEP_CACHE_DIR}:/data",
    "-v", f"{PROCESSED_DIR}:/work",
    "ensemblorg/ensembl-vep",
    "vep",
    "--cache", "--offline",
    "--dir_cache", "/data",
    "--species", "homo_sapiens",
    "--assembly", "GRCh38",
    "--format", "vcf",
    "--tab",
    "--force_overwrite",
    "--fork", "4",
    "--no_stats",
    "--fields", "Uploaded_variation,Location,Allele,Consequence,IMPACT,SYMBOL,Gene,Feature,BIOTYPE,HGVSc,HGVSp,Existing_variation",
    "--input_file", "/work/clinvar_for_vep.vcf",
    "--output_file", "/work/clinvar_vep_consequence.tsv",
]

print("VEP command:")
print(" ".join(vep_command))

if RUN_VEP:
    if not cache_ready:
        raise RuntimeError("VEP cache chua san sang. Chay RUN_CACHE_INSTALL=True truoc khi RUN_VEP=True.")
    run_command(vep_command, check=True)
else:
    print("RUN_VEP=False; chua chay VEP. Doi thanh True sau khi cache da san sang.")

## 5. Parse VEP TSV thanh annotation CSV

VEP co the tra nhieu transcript cho mot variant. Ta chon consequence dai dien theo `IMPACT`:

`HIGH > MODERATE > LOW > MODIFIER`

Neu cung `IMPACT`, giu dong dau tien sau sort.

In [ ]:
def read_vep_tsv(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Chua co VEP output: {path}. Hay chay buoc RUN_VEP=True truoc."
        )

    header = None
    rows = []
    with path.open("r") as handle:
        for line in handle:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith("##"):
                continue
            if line.startswith("#"):
                header = line.lstrip("#").split("\t")
                continue
            rows.append(line.split("\t"))

    if header is None:
        raise ValueError("Khong tim thay header trong VEP TSV")
    if not rows:
        raise ValueError("VEP TSV khong co data rows")

    return pd.DataFrame(rows, columns=header)


if VEP_TSV_PATH.exists():
    vep_df = read_vep_tsv(VEP_TSV_PATH)
    print(vep_df.shape)
    display(vep_df.head())
else:
    print(f"Chua co file {VEP_TSV_PATH}. Bo qua parse cho den khi chay VEP xong.")

In [ ]:
if VEP_TSV_PATH.exists():
    required_vep_cols = ["Uploaded_variation", "Consequence", "IMPACT"]
    missing = [col for col in required_vep_cols if col not in vep_df.columns]
    if missing:
        raise ValueError(f"VEP output thieu cot: {missing}")

    parsed_key = vep_df["Uploaded_variation"].str.extract(
        r"^(?P<Chromosome>[^:]+):(?P<PositionVCF>[^:]+):(?P<ReferenceAlleleVCF>[^:]+):(?P<AlternateAlleleVCF>[^:]+)$"
    )
    if parsed_key.isna().any(axis=None):
        bad_count = int(parsed_key.isna().any(axis=1).sum())
        raise ValueError(
            f"Khong parse duoc Uploaded_variation ID cho {bad_count:,} dong. "
            "Hay kiem tra ID trong VCF dau vao."
        )

    parsed_vep_df = pd.concat([parsed_key, vep_df], axis=1)
    impact_rank = {"HIGH": 0, "MODERATE": 1, "LOW": 2, "MODIFIER": 3}
    parsed_vep_df["impact_rank"] = parsed_vep_df["IMPACT"].map(impact_rank).fillna(99).astype(int)

    selected_vep_df = (
        parsed_vep_df
        .sort_values(["Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF", "impact_rank"])
        .drop_duplicates(subset=key_cols, keep="first")
        .copy()
    )

    annotation_df = selected_vep_df[key_cols + ["Consequence"]].rename(columns={"Consequence": "consequence"})
    annotation_df["gnomAD_AF"] = np.nan
    annotation_df["CADD_PHRED"] = np.nan
    annotation_df["SpliceAI_max"] = np.nan

    annotation_df.to_csv(ANNOTATION_CSV_PATH, index=False)

    print(f"Saved annotation CSV: {ANNOTATION_CSV_PATH}")
    print(f"Rows: {len(annotation_df):,}")
    print("Coverage vs VCF:", f"{len(annotation_df):,} / {len(variant_df):,}")
    display(annotation_df.head())
else:
    print("Chua co VEP output nen chua tao annotation CSV.")

## 6. Kiem tra annotation CSV cho notebook 04

In [ ]:
if ANNOTATION_CSV_PATH.exists():
    check_df = pd.read_csv(ANNOTATION_CSV_PATH)
    expected_cols = [
        "Chromosome", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF",
        "consequence", "gnomAD_AF", "CADD_PHRED", "SpliceAI_max",
    ]
    missing = [col for col in expected_cols if col not in check_df.columns]
    if missing:
        raise ValueError(f"Annotation CSV thieu cot: {missing}")

    print(check_df.shape)
    print("consequence_known:", int(check_df["consequence"].notna().sum()))
    display(check_df["consequence"].value_counts(dropna=False).head(20))
else:
    print(f"Chua co {ANNOTATION_CSV_PATH}. Chay VEP va parse truoc.")

## 7. Chay lai notebook 04

Sau khi `Data/annotations/variant_annotations.csv` ton tai, notebook 04 se tu join cột `consequence`.

Smoke test nhanh:

```bash
jupyter nbconvert --to notebook --execute notebooks/04_xgboost_tabular_baseline.ipynb --output /tmp/04_xgboost_with_consequence.ipynb --ExecutePreprocessor.timeout=600
```

Neu muon test nhanh hon, doi `RUN_FRACTION` trong notebook 04 truoc khi chay.